# Qwen3.8-Flash-Next 125B Gyro-S on Kaggle

Experimental notebook for launching the 125B Gyro-S model on a Kaggle session with **two NVIDIA T4 GPUs**. After launch, use the built-in Gradio browser interface or the optional OpenAI-compatible API.


## Quick start\n\nOpen this notebook in Kaggle from its public raw GitHub URL: [Open Kaggle Notebook Editor](https://www.kaggle.com/notebooks/welcome), then choose **File → Load from URL** and paste:\n\n`https://raw.githubusercontent.com/Thakurdhaval/Qwen3.8-Cyber-AI/main/notebooks/qwen38_flash_next_125b_gyro_s_t4x2.ipynb`\n\n## Before running
1. Enable Internet and an available GPU accelerator in Kaggle notebook settings.
2. Add a Kaggle API token as a Secret named `KAGGLE_API_TOKEN` under Add-ons → Secrets and enable notebook access.
3. Edit the configuration cell with your Kaggle username and a model family/variant supported by Strata.
4. For later sessions, attach your own private model and runtime datasets via Add Input.
5. Read the model license and review the project runtime module before execution.

**Compatibility warning:** this is not a universal Hugging Face loader. Strata supports specific families/variants; run and review the launch plan before downloading large files.

## Why this model needs a special setup

The default is **Qwen3.8-Flash-Next Gyro-S**: a **125B total-parameter sparse MoE model**, with around **6B parameters active per token**. It is still a 125B model in terms of total weights/storage, not a 6B model.

- **Gyro-S main model file:** about 58.5 GB; Strata estimates about **27.6 GB combined RAM + VRAM** for the model weights.
- **Typical hardware in our working test:** 2 × NVIDIA Tesla T4 GPUs (~15 GB VRAM each, around 30 GB combined) plus about 31 GB system RAM.
- **The trick:** Strata's Gyro-S quantization compresses the model and manages expert data across VRAM/system RAM, while a large n-gram table stays on disk and is read as needed. That allows a much larger model to run on less GPU VRAM, with a speed/quality trade-off.
- **Cache size:** the complete model dataset is about 79.94 GiB because it includes support/pack files in addition to the main GGUF. Restoring an attached dataset still transfers/copies those files into the fresh Kaggle runtime; it avoids downloading them from the original model host again.

These are estimates, not guaranteed hardware requirements. Strata supports specific model families/variants only. Review the plan and the [Strata model guide](https://github.com/Niko1221/Strata/blob/main/docs/MODELS.md) before launching.

**Hardware note:** This notebook is configured for a Kaggle T4x2 session. It auto-detects visible GPUs, but the notebook will stop if Kaggle provides fewer than two T4 devices. Actual availability is controlled by Kaggle and can vary by account/session.

In [ ]:
# ===== USER CONFIGURATION: EDIT THIS CELL =====
KAGGLE_USERNAME = "your-kaggle-username"
MODEL_FAMILY = "gyro"
MODEL_VARIANT = "gyro-s"
CONTEXT = "32k"  # Begin with 32k context; do not increase until validated
VISION = "off"
GPU_IDS = ""  # blank = detect all visible NVIDIA GPUs
MODEL_DATA_DIR = "/root/strata-local"

print("Model selection:", MODEL_FAMILY, MODEL_VARIANT)

## 1. Load project runtime and Kaggle credentials

This notebook uses the project-maintained runtime module in this repository. It does not download or execute a separate Kaggle setup-helper repository. The Strata inference engine remains a separate dependency.

In [ ]:
import os, sys, subprocess, json, re
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Qwen3.8-Cyber-AI")
if not (PROJECT_DIR / "app" / "kaggle_runtime.py").is_file():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Thakurdhaval/Qwen3.8-Cyber-AI.git",
        str(PROJECT_DIR)
    ], check=True)

sys.path.insert(0, str(PROJECT_DIR / "app"))
import importlib
import kaggle_runtime as kr
kr = importlib.reload(kr)

KAGGLE_ENV = kr.load_kaggle_credentials(
    KAGGLE_USERNAME,
    secret_name="KAGGLE_API_TOKEN"
)
kr.install_kaggle_cli(KAGGLE_ENV)

HARDWARE = kr.hardware_summary()
print("Detected hardware:", json.dumps(HARDWARE, indent=2))
if len(HARDWARE.get("gpu_names", [])) < 2:
    raise RuntimeError(
        "This 125B Gyro-S profile requires two visible GPUs. Detected: "
        + str(HARDWARE.get("gpu_names", []))
        + ". Stop; do not launch on a single T4."
    )
if any("T4" not in name.upper() for name in HARDWARE.get("gpu_names", [])):
    print("WARNING: This is not the documented 2x Tesla T4 reference configuration. Review memory and compatibility.")



## 2. Prepare the model directory and install Strata

This first-party notebook workflow does not restore or upload private Kaggle cache datasets. Strata will download required model and engine files into the configured directory on the first launch. This can require substantial disk space, bandwidth, and time.

For a fresh Kaggle session, model files may need to be downloaded again. Do not start unless the session has enough available storage and the model license permits your intended use.

In [ ]:
MODEL_ROOT = Path(MODEL_DATA_DIR)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

kr.install_strata_if_missing()
print("Strata installation/version check complete.")
print("Model data directory:", MODEL_ROOT)

## 3. Review the Strata plan

Read the plan output carefully before proceeding. It may identify unsupported model variants, missing dependencies, GPU selection problems, or storage requirements. Stop if the plan fails; do not repeatedly run an unsupported configuration.

In [ ]:
if not GPU_IDS.strip():
    GPU_IDS = kr.default_gpu_list()
print("GPU selection:", GPU_IDS)
if len(HARDWARE.get("gpu_names", [])) < 2:
    raise RuntimeError("Refusing to plan a 125B run without two visible GPUs.")

PLAN_OUTPUT = kr.run_plan(
    family=MODEL_FAMILY,
    variant=MODEL_VARIANT,
    context=CONTEXT,
    vision=VISION,
    gpu_list=GPU_IDS,
    data_dir=MODEL_ROOT,
)

## 4. Launch the authenticated Strata server and Cloudflare tunnel

The helper starts Strata with `--access cloudflare`, waits for a loaded model and API-key protection, then starts a Quick Tunnel if Strata did not create one. Quick Tunnel URLs are temporary and publicly reachable; the Strata API key must remain private.

In [ ]:
import requests

# Start Strata with API-key protection and obtain a temporary HTTPS endpoint.
LAUNCH_OUTPUT = kr.launch_strata(
    family=MODEL_FAMILY,
    variant=MODEL_VARIANT,
    context=CONTEXT,
    vision=VISION,
    gpu_list=GPU_IDS,
    data_dir=MODEL_ROOT,
)

LAUNCH_TUNNEL_URL = kr._extract_tunnel_url(LAUNCH_OUTPUT)
TUNNEL_URL = kr.start_quick_tunnel(LAUNCH_TUNNEL_URL)
HEALTH = kr.wait_health(timeout_seconds=900)
print("Loaded:", HEALTH.get("loaded"))
print("Model:", HEALTH.get("model"))
print("Temporary HTTPS endpoint:", TUNNEL_URL)

## 5. Verify the public API and run a short generation

The next cell checks that unauthenticated API requests are rejected with HTTP 401/403, authenticated requests can list models, and a short authenticated chat completion succeeds. It never prints the Strata API key.

In [ ]:
TEST_RESULT = kr.verify_public_endpoint(TUNNEL_URL)
MODEL_ID = TEST_RESULT["model_id"]
print("End-to-end authenticated API test: PASS")
print("Model ID for OpenCode:", MODEL_ID)

## 6. Cache behavior

This project-owned notebook currently does not upload or restore model/runtime caches to Kaggle datasets. The first launch downloads required files, and a new Kaggle session may need to download them again. Keep sufficient disk space and bandwidth available.


In [ ]:
print("Private model/runtime cache upload is not included in this project-owned runtime yet.")
print("The model and runtime may need to be downloaded again in a new Kaggle session.")

## 6. ChatGPT-style browser GUI\n\nAfter the authenticated API test passes, run the next cell to start a Gradio chat interface. It connects to the local Strata API; your Strata key stays in the Kaggle runtime. The temporary Gradio share URL is protected by a username and password that you choose at runtime. Keep them private and close the session when finished.\n

In [ ]:
# Install the lightweight browser UI package (the model is already served by Strata).%pip -q install gradioimport getpassimport gradio as grimport requestsSTRATA_LOCAL_URL = "http://127.0.0.1:8080/v1"STRATA_API_KEY = kr.read_api_key()  # Read privately; never print or store in notebook output.SOC_SYSTEM_PROMPT = (PROJECT_DIR / "config" / "soc_system_prompt.txt").read_text(encoding="utf-8")GUI_USERNAME = input("Choose a private GUI username: ").strip()GUI_PASSWORD = getpass.getpass("Choose a strong GUI password (not your Kaggle password): ")if not GUI_USERNAME or len(GUI_PASSWORD) < 12:    raise ValueError("Use a non-empty username and a GUI password of at least 12 characters.")def chat_with_gyro(message, history):    messages = [{"role": "system", "content": SOC_SYSTEM_PROMPT}]    for item in history or []:        if isinstance(item, dict) and item.get("role") in ("user", "assistant"):            content = item.get("content")            if isinstance(content, str):                messages.append({"role": item["role"], "content": content})    messages.append({"role": "user", "content": message})    response = requests.post(        STRATA_LOCAL_URL + "/chat/completions",        headers={"Authorization": "Bearer " + STRATA_API_KEY, "Content-Type": "application/json"},        json={"model": MODEL_ID, "messages": messages, "max_tokens": 1024, "stream": False},        timeout=300,    )    response.raise_for_status()    payload = response.json()    return payload["choices"][0]["message"]["content"]chat_ui = gr.ChatInterface(    fn=chat_with_gyro,    title="Qwen3.8-Cyber-AI · SOC Assistant",    description="Experimental defensive SOC assistant for alert triage, log analysis, incident response, and detection learning. Do not enter secrets or confidential data.",    examples=[        "Triage this synthetic Windows failed-logon alert and identify what evidence is missing: Event ID 4625, account=analyst1, source_ip=198.51.100.24, time=2026-11-04T08:30:00Z.",        "Explain these Linux authentication log fields and list safe read-only investigation steps.",        "Create a defensive detection concept for repeated failed logons. Include false positives and a safe test plan.",        "Explain the SOC incident-response lifecycle using a synthetic phishing alert.",    ],)chat_ui.launch(    share=True,    auth=(GUI_USERNAME, GUI_PASSWORD),    server_name="0.0.0.0",    server_port=7860,    prevent_thread_lock=True,)

## 7. Connect OpenCode Desktop

Merge the printed provider entry into your existing OpenCode configuration; do not replace unrelated providers. Retrieve the Strata API key privately in the Kaggle shell with `strata launch --show-key`. Save it locally on your Windows PC at `~/.config/opencode/strata-key.txt`. Never commit or share the key.

Keep this Kaggle session and tunnel running while using the model. The URL may change after a restart.

In [ ]:
print("OpenCode baseURL:", TUNNEL_URL.rstrip("/") + "/v1")
print("OpenCode model ID:", MODEL_ID)
print("\nMerge this provider entry inside the existing top-level providers object:")
print(kr.opencode_config_snippet(TUNNEL_URL, MODEL_ID))
print("\nRetrieve the API key privately in the Kaggle terminal; do not print it in notebook output:")
print("strata launch --show-key")
print("Store it locally at ~/.config/opencode/qwen38-strata-key.txt")

## 8. Keep the notebook actively supervising Strata (optional, 6 hours)

Run this cell after the API test succeeds. It keeps an actual health-monitoring task running while checking the local Strata service and the tunnel's authentication boundary. It does **not** bypass Kaggle's maximum runtime or guarantee uptime; the session, GPU quota, network, and tunnel can still be stopped by Kaggle or fail. Stop this cell when you're done.

In [ ]:
import time
from datetime import datetime, timezone

# Set to 6 for a six-hour supervision window. Do not exceed Kaggle's session limits.
MONITOR_HOURS = 6
HEALTH_CHECK_SECONDS = 60
REPORT_EVERY_SECONDS = 300

monitor_started = time.monotonic()
monitor_deadline = monitor_started + MONITOR_HOURS * 3600
last_report = 0.0
last_state = None
print(f"Monitoring Strata for up to {MONITOR_HOURS} hours. Keep Kaggle session within its runtime/GPU quota.", flush=True)

while time.monotonic() < monitor_deadline:
    now_mono = time.monotonic()
    remaining = max(0, int(monitor_deadline - now_mono))
    try:
        response = requests.get("http://127.0.0.1:8080/health", timeout=10)
        payload = response.json() if response.ok else {}
        healthy = bool(response.ok and payload.get("status") == "ok" and payload.get("loaded") is True and payload.get("api_key") is True)
        state = "HEALTHY" if healthy else f"NOT READY (HTTP {response.status_code}; loaded={payload.get('loaded')}; api_key={payload.get('api_key')})"
    except Exception as exc:
        healthy = False
        state = "HEALTH CHECK ERROR: " + str(exc)[:180]

    # Occasionally make sure the public endpoint is still reachable and rejects no-key requests.
    if healthy and (now_mono - last_report >= REPORT_EVERY_SECONDS or last_state != state):
        try:
            public_check = requests.get(TUNNEL_URL.rstrip("/") + "/v1/models", timeout=15)
            if public_check.status_code not in (401, 403):
                state += f" | WARNING: public no-key check returned HTTP {public_check.status_code} (expected 401/403)"
                healthy = False
            else:
                state += f" | tunnel/auth boundary OK (HTTP {public_check.status_code} without key)"
        except Exception as exc:
            state += " | tunnel check failed: " + str(exc)[:160]
            healthy = False

    if now_mono - last_report >= REPORT_EVERY_SECONDS or state != last_state:
        print(f"[{datetime.now().astimezone().isoformat(timespec='seconds')}] {state} | {remaining//3600:02d}:{(remaining%3600)//60:02d}:{remaining%60:02d} remaining", flush=True)
        last_report = now_mono
        last_state = state
    time.sleep(HEALTH_CHECK_SECONDS)

print("Supervision window complete. Strata may continue until Kaggle stops the session; check health before use.", flush=True)

> **Important:** A running monitor cell is a legitimate health monitor, not an uptime guarantee. Kaggle may end a session at its hard limit, when quotas are exhausted, or if the runtime/server fails. For a reliable always-on endpoint, use infrastructure designed for persistent services.